# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset schema is accessible via the following Croissant JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure mlcroissant is installed (uncomment if running in Colab or fresh environment)
!pip install -U mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # Optional: to avoid warning messages in notebook output

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Show dataset-level metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s in this dataset.

In [ ]:
# Examine all record sets by their @id

record_sets = dataset.record_sets
if record_sets:
    print("Available record sets:")
    for rs in record_sets:
        print(f"  - @id: {rs['@id']} (name: {rs.get('name', 'Unnamed')}, description: {rs.get('description', '')})")
else:
    print("No record sets found in dataset.metadata.record_sets.\nAttempting to infer record sets from the dataset.")
    # Try listing records with an empty identifier (should yield an error if none exists)
    # mlcroissant exposes available record_set @ids via dataset._record_sets internally
    # Let's show those (not a public API, but for interactive exploration, ok)
    record_set_ids = list(dataset._record_sets.keys())
    if record_set_ids:
        print("Record sets loaded by Croissant schema:")
        for rs_id in record_set_ids:
            print(f"  - @id: {rs_id}")
    else:
        print("Could not infer record sets. Please check the dataset schema for record set definition.")

For further exploration, we'll enumerate the fields (or columns) of the first non-empty record set, referencing every field or column by its `@id`.

In [ ]:
# Choose a record set to explore
if record_sets:
    record_set_id = record_sets[0]['@id']
else:
    record_set_ids = list(dataset._record_sets.keys())
    if record_set_ids:
        record_set_id = record_set_ids[0]
    else:
        raise ValueError("No record sets found in the Croissant schema.")

print(f"\nExploring fields/columns in record set @id: {record_set_id}")
fields = dataset._record_sets[record_set_id]['fields'] if 'fields' in dataset._record_sets[record_set_id] else []
if fields:
    for field in fields:
        print(f"  - Field @id: {field['@id']} (name: {field.get('name', 'Unnamed')}, data type: {field.get('data_type', 'Unknown')})")
else:
    print("No explicit fields listed. Attempting to read a few records to infer column names.")
    sample_records = list(dataset.records(record_set=record_set_id))
    if sample_records:
        print("Sample record keys:")
        print(list(sample_records[0].keys()))
    else:
        print("Could not extract fields or records for this record set.")

Let's print a few records from the identified record set for direct inspection, referencing every key by its original Croissant `@id` (column/field names).

In [ ]:
# Show a sample of records from the selected record set, using its @id
print(f"\nSample records for record set @id: {record_set_id} (first 2 shown):")
sample_records = list(dataset.records(record_set=record_set_id))
for ix, rec in enumerate(sample_records[:2]):
    print(f"Record {ix}: {json.dumps(rec, indent=2)}")

## 3. Data Extraction
Load data from all available record sets into DataFrames for further analysis.

In [ ]:
# Gather all available record set @id's
all_record_set_ids = list(dataset._record_sets.keys())

dataframes = {}
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {rs_id}")
    else:
        print(f"No records loaded for record set @id: {rs_id}")

# Use the primary record set for further EDA
primary_record_set = record_set_id
if primary_record_set in dataframes and not dataframes[primary_record_set].empty:
    print(f"\nColumns in primary record set (@id: {primary_record_set}):")
    print(dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())
else:
    print("Primary record set missing or empty. Please check the Croissant schema and data files.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate typical data processing: filtering, normalization, grouping, always referencing Croissant `@id`s.

In [ ]:
# Identify a numeric field (by @id) from the DataFrame for numeric EDA.
df = dataframes[primary_record_set]

# Let’s try to automatically select a numeric field from the columns
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    print("No obvious numeric fields found. Attempting to coerce columns to float for demonstration.")
    # Just for demonstration, pick the first column and try casting
    numeric_field_id = df.columns[0]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    print(f"Using field @id: {numeric_field_id} (after coercing to numeric)")

# Set a threshold for filtering - here arbitrarily 10; adjust as suits the field's range
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a categorical/grouping field (by @id) to group the data
group_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    print(f"Grouping by field @id: {group_field_id}")
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    )
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No string/categorical fields found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship with the chosen group field (if applicable), referencing all axes/legends by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id} (@id)')
plt.xlabel(f'{numeric_field_id} (@id)')
plt.ylabel('Frequency')
plt.show()

# If grouping field available, make a boxplot
if group_field_candidates:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
    plt.xlabel(f'{group_field_id} (@id)')
    plt.ylabel(f'{numeric_field_id} (@id)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR² dataset via its Croissant JSON-LD schema using `mlcroissant`;
- Explored dataset metadata and structure, referencing all elements by their `@id`;
- Listed record sets and fields available for analysis;
- Loaded records into Pandas DataFrames and performed basic EDA (filtering, normalization, grouping);
- Visualized numeric field distributions and group comparisons by original Croissant `@id`.

This workflow demonstrates a reproducible FAIR data practice, enables transparent referencing of all dataset elements using their globally unique identifiers, and prepares your tabular data for deeper domain analysis.
